In [9]:
#!/usr/bin/env python3
"""
Gaussian Blur: Complete Comparison
Separable Convolution vs Full 2D Convolution

This standalone application demonstrates both approaches and verifies they produce
identical results while showing the performance difference.
"""

import numpy as np
from scipy import ndimage
import time
import argparse


def gaussian_2d_kernel(sigma=1.0, kernel_size=5):
    """
    Create 2D Gaussian kernel

    Args:
        sigma: Standard deviation of Gaussian
        kernel_size: Size of kernel (must be odd)

    Returns:
        2D numpy array of shape (kernel_size, kernel_size)
    """
    if kernel_size % 2 == 0:
        kernel_size += 1  # Ensure odd size

    x = np.arange(kernel_size) - kernel_size // 2
    y = np.arange(kernel_size) - kernel_size // 2
    x, y = np.meshgrid(x, y)

    # 2D Gaussian formula: G(x,y) = (1/(2πσ²)) * exp(-(x²+y²)/(2σ²))
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()  # Normalize

    return kernel


def gaussian_1d_kernel(sigma=1.0, kernel_size=5):
    """
    Create 1D Gaussian kernel

    Args:
        sigma: Standard deviation of Gaussian
        kernel_size: Size of kernel (must be odd)

    Returns:
        1D numpy array of shape (kernel_size,)
    """
    if kernel_size % 2 == 0:
        kernel_size += 1

    x = np.arange(kernel_size) - kernel_size // 2

    # 1D Gaussian formula: G(x) = (1/√(2πσ²)) * exp(-x²/(2σ²))
    kernel = np.exp(-x**2 / (2 * sigma**2))
    kernel = kernel / kernel.sum()  # Normalize

    return kernel


def gaussian_2d_full(image, sigma=1.0, kernel_size=5):
    """
    Apply Gaussian blur using full 2D convolution

    Args:
        image: Input image (2D numpy array)
        sigma: Standard deviation
        kernel_size: Size of kernel

    Returns:
        Blurred image
    """
    kernel = gaussian_2d_kernel(sigma, kernel_size)
    return ndimage.convolve(image, kernel, mode='reflect')


def gaussian_separable(image, sigma=1.0, kernel_size=5):
    """
    Apply Gaussian blur using separable convolution (two 1D passes)

    Args:
        image: Input image (2D numpy array)
        sigma: Standard deviation
        kernel_size: Size of kernel

    Returns:
        Blurred image
    """
    kernel_1d = gaussian_1d_kernel(sigma, kernel_size)

    # First pass: convolve along rows (horizontal)
    temp = ndimage.convolve1d(image, kernel_1d, axis=1, mode='reflect')

    # Second pass: convolve along columns (vertical)
    result = ndimage.convolve1d(temp, kernel_1d, axis=0, mode='reflect')

    return result


def create_test_image(width=1000, height=1000, pattern='gradient'):
    """
    Create a test image with various patterns

    Args:
        width: Image width
        height: Image height
        pattern: Type of pattern ('random', 'gradient', 'checkerboard', 'gaussian')

    Returns:
        2D numpy array
    """
    if pattern == 'random':
        return np.random.rand(height, width)  # float64 by default

    elif pattern == 'gradient':
        x = np.linspace(0, 1, width)
        y = np.linspace(0, 1, height)
        X, Y = np.meshgrid(x, y)
        return (X + Y) / 2

    elif pattern == 'checkerboard':
        image = np.zeros((height, width), dtype=np.float32)
        square_size = 32
        for i in range(0, height, square_size):
            for j in range(0, width, square_size):
                if ((i // square_size) + (j // square_size)) % 2 == 0:
                    image[i:i+square_size, j:j+square_size] = 1.0
        return image

    elif pattern == 'gaussian':
        # Create a Gaussian blob in the center
        x = np.arange(width) - width // 2
        y = np.arange(height) - height // 2
        X, Y = np.meshgrid(x, y)
        return np.exp(-(X**2 + Y**2) / (2 * (min(width, height) / 6)**2))

    else:
        raise ValueError(f"Unknown pattern: {pattern}")


def benchmark_method(method_func, image, sigma, kernel_size, iterations=10):
    """
    Benchmark a blur method

    Args:
        method_func: Function to benchmark
        image: Input image
        sigma: Gaussian sigma
        kernel_size: Kernel size
        iterations: Number of iterations

    Returns:
        Dictionary with timing statistics
    """
    times = []

    # Warm-up run
    result = method_func(image, sigma, kernel_size)

    # Benchmark runs
    for _ in range(iterations):
        start = time.time()
        result = method_func(image, sigma, kernel_size)
        end = time.time()
        times.append((end - start) * 1000)  # Convert to ms

    return {
        'result': result,
        'min': min(times),
        'max': max(times),
        'mean': np.mean(times),
        'std': np.std(times),
        'median': np.median(times)
    }


def verify_results(result1, result2, tolerance=None):
    """
    Verify that two results are identical within tolerance

    Args:
        result1: First result
        result2: Second result
        tolerance: Maximum acceptable difference (auto-determined if None)

    Returns:
        Dictionary with verification statistics
    """
    diff = np.abs(result1 - result2)

    # Auto-determine appropriate tolerance based on data type and operations
    if tolerance is None:
        dtype = result1.dtype
        if dtype == np.float32:
            # For float32: ~1000× machine epsilon accounts for:
            # - Accumulated rounding errors from ~100 operations
            # - Different computation order (non-associativity)
            # - Intermediate storage
            tolerance = 1000 * np.finfo(np.float32).eps  # ~1.2e-4
        else:  # float64
            # For float64: ~1000× machine epsilon
            # Still need to account for different computation paths
            tolerance = 1000 * np.finfo(np.float64).eps  # ~2.2e-13

    return {
        'max_diff': np.max(diff),
        'mean_diff': np.mean(diff),
        'identical': np.max(diff) < tolerance,
        'tolerance': tolerance
    }


def print_kernel_info(sigma, kernel_size):
    """Print kernel information"""
    kernel_1d = gaussian_1d_kernel(sigma, kernel_size)
    kernel_2d = gaussian_2d_kernel(sigma, kernel_size)

    print(f"\n{'='*70}")
    print(f"Kernel Information (σ={sigma}, size={kernel_size}x{kernel_size})")
    print(f"{'='*70}")

    print(f"\n1D Kernel ({kernel_size} elements):")
    print(f"  Values: {kernel_1d}")
    print(f"  Sum: {kernel_1d.sum():.10f} (should be 1.0)")

    print(f"\n2D Kernel ({kernel_size}x{kernel_size} elements):")
    print(f"  Center row: {kernel_2d[kernel_size//2, :]}")
    print(f"  Sum: {kernel_2d.sum():.10f} (should be 1.0)")

    # Verify separability
    outer_product = np.outer(kernel_1d, kernel_1d)
    separable_diff = np.max(np.abs(kernel_2d - outer_product))
    print(f"\n  Separability check:")
    print(f"    Max difference between 2D and outer(1D,1D): {separable_diff:.2e}")
    print(f"    Kernel is separable: {separable_diff < 1e-10}")


def print_benchmark_results(name, stats, image_size, kernel_size):
    """Print benchmark statistics"""
    height, width = image_size
    pixels = width * height
    throughput = (pixels / 1e6) / (stats['mean'] / 1000.0)
    ops_per_pixel = kernel_size * kernel_size if '2D' in name else 2 * kernel_size

    print(f"\n{name}:")
    print(f"  Time: {stats['mean']:.3f} ± {stats['std']:.3f} ms")
    print(f"  Range: [{stats['min']:.3f}, {stats['max']:.3f}] ms")
    print(f"  Throughput: {throughput:.2f} Mpixels/sec")
    print(f"  Operations/pixel: {ops_per_pixel}")


def save_output(filename, image):
    """Save image to file (as numpy array)"""
    np.save(filename, image)
    print(f"\nSaved output to: {filename}")


def main():
    parser = argparse.ArgumentParser(
        description='Gaussian Blur Comparison: Separable vs Full 2D Convolution'
    )
    parser.add_argument('--width', type=int, default=1000,
                        help='Image width (default: 1000)')
    parser.add_argument('--height', type=int, default=1000,
                        help='Image height (default: 1000)')
    parser.add_argument('--sigma', type=float, default=2.0,
                        help='Gaussian sigma (default: 2.0)')
    parser.add_argument('--kernel-size', type=int, default=11,
                        help='Kernel size (default: 11)')
    parser.add_argument('--pattern', type=str, default='random',
                        choices=['random', 'gradient', 'checkerboard', 'gaussian'],
                        help='Test pattern (default: random)')
    parser.add_argument('--iterations', type=int, default=10,
                        help='Benchmark iterations (default: 10)')
    parser.add_argument('--save', action='store_true',
                        help='Save output images')

    args = parser.parse_args([])

    # Header
    print("="*70)
    print("Gaussian Blur: Separable vs Full 2D Convolution")
    print("="*70)

    # Configuration
    print(f"\nConfiguration:")
    print(f"  Image size: {args.width} x {args.height}")
    print(f"  Kernel size: {args.kernel_size} x {args.kernel_size}")
    print(f"  Sigma: {args.sigma}")
    print(f"  Pattern: {args.pattern}")
    print(f"  Benchmark iterations: {args.iterations}")

    # Create test image
    print(f"\nCreating test image ({args.pattern} pattern)...")
    image = create_test_image(args.width, args.height, args.pattern)

    # Print kernel information
    print_kernel_info(args.sigma, args.kernel_size)

    # Benchmark Full 2D Convolution
    print(f"\n{'='*70}")
    print("Benchmarking Full 2D Convolution")
    print(f"{'='*70}")
    stats_2d = benchmark_method(gaussian_2d_full, image, args.sigma,
                                args.kernel_size, args.iterations)
    print_benchmark_results("Full 2D Convolution", stats_2d,
                           (args.height, args.width), args.kernel_size)

    # Benchmark Separable Convolution
    print(f"\n{'='*70}")
    print("Benchmarking Separable Convolution")
    print(f"{'='*70}")
    stats_sep = benchmark_method(gaussian_separable, image, args.sigma,
                                 args.kernel_size, args.iterations)
    print_benchmark_results("Separable Convolution", stats_sep,
                           (args.height, args.width), args.kernel_size)

    # Verify results are identical
    print(f"\n{'='*70}")
    print("Verification")
    print(f"{'='*70}")
    verification = verify_results(stats_2d['result'], stats_sep['result'])

    print(f"\nResult comparison:")
    print(f"  Max difference: {verification['max_diff']:.2e}")
    print(f"  Mean difference: {verification['mean_diff']:.2e}")
    print(f"  Results identical (tolerance={verification['tolerance']:.2e}): "
          f"{verification['identical']}")

    # Performance comparison
    print(f"\n{'='*70}")
    print("Performance Summary")
    print(f"{'='*70}")

    speedup = stats_2d['mean'] / stats_sep['mean']
    theoretical_speedup = args.kernel_size / 2

    print(f"\nSpeedup (Separable vs Full 2D):")
    print(f"  Actual: {speedup:.2f}x")
    print(f"  Theoretical: {theoretical_speedup:.2f}x")
    print(f"  Efficiency: {(speedup / theoretical_speedup) * 100:.1f}%")

    # Sample output values
    print(f"\n{'='*70}")
    print("Sample Output Values (center 5x5 region)")
    print(f"{'='*70}")

    cy, cx = args.height // 2, args.width // 2
    print("\nInput:")
    print(image[cy-2:cy+3, cx-2:cx+3])

    print("\nOutput (both methods produce identical results):")
    print(stats_sep['result'][cy-2:cy+3, cx-2:cx+3])

    # Save outputs if requested
    if args.save:
        save_output('input.npy', image)
        save_output('output_2d.npy', stats_2d['result'])
        save_output('output_separable.npy', stats_sep['result'])

    # Test multiple kernel sizes
    print(f"\n{'='*70}")
    print("Performance Scaling with Kernel Size")
    print(f"{'='*70}")

    print(f"\n{'Kernel':<10} {'Size':<8} {'2D Time':<12} {'Sep Time':<12} {'Speedup':<10} {'Theoretical'}")
    print("-" * 70)

    for ksize in [3, 5, 7, 9, 11, 15, 21]:
        if ksize > args.kernel_size:
            break

        # Quick benchmark (fewer iterations)
        stats_2d_test = benchmark_method(gaussian_2d_full, image, args.sigma,
                                         ksize, iterations=3)
        stats_sep_test = benchmark_method(gaussian_separable, image, args.sigma,
                                          ksize, iterations=3)

        speedup_test = stats_2d_test['mean'] / stats_sep_test['mean']
        theoretical = ksize / 2

        print(f"{ksize}x{ksize:<6} {ksize*ksize:<8} "
              f"{stats_2d_test['mean']:>10.2f}ms "
              f"{stats_sep_test['mean']:>10.2f}ms "
              f"{speedup_test:>8.2f}x "
              f"{theoretical:>10.2f}x")

    print(f"\n{'='*70}")
    print("Completed Successfully!")
    print(f"{'='*70}\n")


if __name__ == "__main__":
    main()


Gaussian Blur: Separable vs Full 2D Convolution

Configuration:
  Image size: 1000 x 1000
  Kernel size: 11 x 11
  Sigma: 2.0
  Pattern: random
  Benchmark iterations: 10

Creating test image (random pattern)...

Kernel Information (σ=2.0, size=11x11)

1D Kernel (11 elements):
  Values: [0.00881223 0.02714358 0.06511406 0.12164907 0.17699836 0.20056541
 0.17699836 0.12164907 0.06511406 0.02714358 0.00881223]
  Sum: 1.0000000000 (should be 1.0)

2D Kernel (11x11 elements):
  Center row: [0.00176743 0.00544406 0.01305963 0.0243986  0.03549975 0.04022649
 0.03549975 0.0243986  0.01305963 0.00544406 0.00176743]
  Sum: 1.0000000000 (should be 1.0)

  Separability check:
    Max difference between 2D and outer(1D,1D): 6.94e-18
    Kernel is separable: True

Benchmarking Full 2D Convolution

Full 2D Convolution:
  Time: 155.608 ± 5.437 ms
  Range: [148.886, 164.323] ms
  Throughput: 6.43 Mpixels/sec
  Operations/pixel: 121

Benchmarking Separable Convolution

Separable Convolution:
  Time: 17

In [1]:
%%writefile gaussian_blur_cuda.cu
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <stdio.h>
#include <math.h>

#define KERNEL_RADIUS 5
#define KERNEL_SIZE (2 * KERNEL_RADIUS + 1)
#define BLOCK_SIZE 16

// Error checking macro
#define CUDA_CHECK(call) \
    do { \
        cudaError_t error = call; \
        if (error != cudaSuccess) { \
            fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, \
                    cudaGetErrorString(error)); \
            exit(EXIT_FAILURE); \
        } \
    } while(0)

/*
 * Generate 1D Gaussian kernel
 * Formula: G(x) = (1/sqrt(2*pi*sigma^2)) * exp(-x^2 / (2*sigma^2))
 */
__host__ void generateGaussianKernel(float* kernel, int radius, float sigma) {
    float sum = 0.0f;

    for (int i = -radius; i <= radius; i++) {
        float value = expf(-(i * i) / (2.0f * sigma * sigma));
        kernel[i + radius] = value;
        sum += value;
    }

    // Normalize
    for (int i = 0; i < 2 * radius + 1; i++) {
        kernel[i] /= sum;
    }
}

/*
 * Horizontal Gaussian blur kernel
 * Each thread processes one pixel, reading from shared memory
 */
__global__ void gaussianBlurHorizontal(
    const float* input,
    float* output,
    const float* kernel,
    int width,
    int height,
    int radius
) {
    // Shared memory for the row (includes halo region)
    extern __shared__ float sharedRow[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (y >= height) return;

    // Load data into shared memory with halo
    int sharedIdx = threadIdx.x + radius;

    // Load main data
    if (x < width) {
        sharedRow[sharedIdx] = input[y * width + x];
    }

    // Load left halo
    if (threadIdx.x < radius) {
        int leftX = blockIdx.x * blockDim.x - radius + threadIdx.x;
        if (leftX < 0) {
            sharedRow[threadIdx.x] = input[y * width + 0]; // Clamp
        } else {
            sharedRow[threadIdx.x] = input[y * width + leftX];
        }
    }

    // Load right halo
    if (threadIdx.x >= blockDim.x - radius) {
        int rightX = blockIdx.x * blockDim.x + threadIdx.x + radius;
        int sharedRightIdx = threadIdx.x + 2 * radius;
        if (rightX >= width) {
            sharedRow[sharedRightIdx] = input[y * width + width - 1]; // Clamp
        } else {
            sharedRow[sharedRightIdx] = input[y * width + rightX];
        }
    }

    __syncthreads();

    // Perform convolution
    if (x < width) {
        float sum = 0.0f;

        for (int k = -radius; k <= radius; k++) {
            sum += sharedRow[sharedIdx + k] * kernel[k + radius];
        }

        output[y * width + x] = sum;
    }
}

/*
 * Vertical Gaussian blur kernel
 * Each thread processes one pixel, reading columns
 */
__global__ void gaussianBlurVertical(
    const float* input,
    float* output,
    const float* kernel,
    int width,
    int height,
    int radius
) {
    // Shared memory for the column (includes halo region)
    extern __shared__ float sharedCol[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width) return;

    // Load data into shared memory with halo
    int sharedIdx = threadIdx.y + radius;

    // Load main data
    if (y < height) {
        sharedCol[sharedIdx] = input[y * width + x];
    }

    // Load top halo
    if (threadIdx.y < radius) {
        int topY = blockIdx.y * blockDim.y - radius + threadIdx.y;
        if (topY < 0) {
            sharedCol[threadIdx.y] = input[0 * width + x]; // Clamp
        } else {
            sharedCol[threadIdx.y] = input[topY * width + x];
        }
    }

    // Load bottom halo
    if (threadIdx.y >= blockDim.y - radius) {
        int bottomY = blockIdx.y * blockDim.y + threadIdx.y + radius;
        int sharedBottomIdx = threadIdx.y + 2 * radius;
        if (bottomY >= height) {
            sharedCol[sharedBottomIdx] = input[(height - 1) * width + x]; // Clamp
        } else {
            sharedCol[sharedBottomIdx] = input[bottomY * width + x];
        }
    }

    __syncthreads();

    // Perform convolution
    if (y < height) {
        float sum = 0.0f;

        for (int k = -radius; k <= radius; k++) {
            sum += sharedCol[sharedIdx + k] * kernel[k + radius];
        }

        output[y * width + x] = sum;
    }
}

/*
 * Apply separable Gaussian blur
 */
void applySeparableGaussianBlur(
    const float* d_input,
    float* d_output,
    float* d_temp,
    const float* d_kernel,
    int width,
    int height,
    int radius
) {
    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridSize(
        (width + blockSize.x - 1) / blockSize.x,
        (height + blockSize.y - 1) / blockSize.y
    );

    // Shared memory size (block size + 2 * radius for halo)
    int sharedMemSize = (BLOCK_SIZE + 2 * radius) * sizeof(float);

    // Horizontal pass
    gaussianBlurHorizontal<<<gridSize, blockSize, sharedMemSize>>>(
        d_input, d_temp, d_kernel, width, height, radius
    );
    CUDA_CHECK(cudaGetLastError());

    // Vertical pass
    gaussianBlurVertical<<<gridSize, blockSize, sharedMemSize>>>(
        d_temp, d_output, d_kernel, width, height, radius
    );
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaDeviceSynchronize());
}

/*
 * Main function demonstrating usage
 */
int main() {
    // Image dimensions
    const int width = 1920;
    const int height = 1080;
    const int imageSize = width * height * sizeof(float);

    // Gaussian parameters
    const int radius = KERNEL_RADIUS;
    const float sigma = 2.0f;
    const int kernelSize = 2 * radius + 1;

    printf("Separable Gaussian Blur (CUDA)\n");
    printf("Image size: %d x %d\n", width, height);
    printf("Kernel radius: %d (size: %d)\n", radius, kernelSize);
    printf("Sigma: %.2f\n\n", sigma);

    // Allocate host memory
    float* h_input = (float*)malloc(imageSize);
    float* h_output = (float*)malloc(imageSize);
    float* h_kernel = (float*)malloc(kernelSize * sizeof(float));

    // Initialize input with test pattern
    for (int i = 0; i < width * height; i++) {
        h_input[i] = (float)(rand() % 256) / 255.0f;
    }

    // Generate Gaussian kernel
    generateGaussianKernel(h_kernel, radius, sigma);

    printf("Gaussian kernel coefficients:\n");
    for (int i = 0; i < kernelSize; i++) {
        printf("%.6f ", h_kernel[i]);
    }
    printf("\n\n");

    // Allocate device memory
    float *d_input, *d_output, *d_temp, *d_kernel;
    CUDA_CHECK(cudaMalloc(&d_input, imageSize));
    CUDA_CHECK(cudaMalloc(&d_output, imageSize));
    CUDA_CHECK(cudaMalloc(&d_temp, imageSize));
    CUDA_CHECK(cudaMalloc(&d_kernel, kernelSize * sizeof(float)));

    // Copy data to device
    CUDA_CHECK(cudaMemcpy(d_input, h_input, imageSize, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_kernel, h_kernel, kernelSize * sizeof(float), cudaMemcpyHostToDevice));

    // Create CUDA events for timing
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Warm-up run
    applySeparableGaussianBlur(d_input, d_output, d_temp, d_kernel, width, height, radius);

    // Timed run
    CUDA_CHECK(cudaEventRecord(start));
    applySeparableGaussianBlur(d_input, d_output, d_temp, d_kernel, width, height, radius);
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float milliseconds = 0;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start, stop));

    printf("Execution time: %.3f ms\n", milliseconds);
    printf("Throughput: %.2f Mpixels/sec\n", (width * height / 1e6) / (milliseconds / 1000.0f));

    // Copy result back to host
    CUDA_CHECK(cudaMemcpy(h_output, d_output, imageSize, cudaMemcpyDeviceToHost));

    // Verify result (check a few pixels)
    printf("\nSample output values:\n");
    for (int i = 0; i < 5; i++) {
        int idx = (height / 2) * width + (width / 2) + i;
        printf("Pixel[%d][%d]: %.6f\n", height / 2, width / 2 + i, h_output[idx]);
    }

    // Cleanup
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_input));
    CUDA_CHECK(cudaFree(d_output));
    CUDA_CHECK(cudaFree(d_temp));
    CUDA_CHECK(cudaFree(d_kernel));
    free(h_input);
    free(h_output);
    free(h_kernel);

    printf("\nGaussian blur completed successfully!\n");

    return 0;
}

Writing gaussian_blur_cuda.cu


In [3]:
!nvcc -I /usr/local/cuda/samples/common/inc/ -L/usr/local/cuda/include -lcublas -lcusolver -arch=sm_75 -Wno-deprecated-gpu-targets gaussian_blur_cuda.cu

In [4]:
!./a.out

Separable Gaussian Blur (CUDA)
Image size: 1920 x 1080
Kernel radius: 5 (size: 11)
Sigma: 2.00

Gaussian kernel coefficients:
0.008812 0.027144 0.065114 0.121649 0.176998 0.200565 0.176998 0.121649 0.065114 0.027144 0.008812 

Execution time: 0.401 ms
Throughput: 5172.00 Mpixels/sec

Sample output values:
Pixel[540][960]: 0.548679
Pixel[540][961]: 0.548679
Pixel[540][962]: 0.548679
Pixel[540][963]: 0.548679
Pixel[540][964]: 0.548679

Gaussian blur completed successfully!


In [5]:
%%writefile gaussian_blur_kernels.cu
/*
 * CUDA Kernels Implementation
 * Separable Gaussian Blur
 */

#include <cuda_runtime.h>
#include <device_launch_parameters.h>

/*
 * Horizontal Gaussian blur kernel with shared memory optimization
 */
__global__ void gaussianBlurHorizontalKernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    const float* __restrict__ kernel,
    int width,
    int height,
    int radius
) {
    extern __shared__ float sharedRow[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (y >= height) return;

    int sharedIdx = threadIdx.x + radius;

    // Load center data
    if (x < width) {
        sharedRow[sharedIdx] = input[y * width + x];
    }

    // Load left halo
    if (threadIdx.x < radius) {
        int leftX = blockIdx.x * blockDim.x - radius + threadIdx.x;
        sharedRow[threadIdx.x] = (leftX < 0) ?
            input[y * width] : input[y * width + leftX];
    }

    // Load right halo
    if (threadIdx.x >= blockDim.x - radius) {
        int rightX = blockIdx.x * blockDim.x + threadIdx.x + radius;
        int sharedRightIdx = threadIdx.x + 2 * radius;
        sharedRow[sharedRightIdx] = (rightX >= width) ?
            input[y * width + width - 1] : input[y * width + rightX];
    }

    __syncthreads();

    // Perform convolution
    if (x < width) {
        float sum = 0.0f;

        #pragma unroll
        for (int k = -radius; k <= radius; k++) {
            sum += sharedRow[sharedIdx + k] * kernel[k + radius];
        }

        output[y * width + x] = sum;
    }
}

/*
 * Vertical Gaussian blur kernel with shared memory optimization
 */
__global__ void gaussianBlurVerticalKernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    const float* __restrict__ kernel,
    int width,
    int height,
    int radius
) {
    extern __shared__ float sharedCol[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width) return;

    int sharedIdx = threadIdx.y + radius;

    // Load center data
    if (y < height) {
        sharedCol[sharedIdx] = input[y * width + x];
    }

    // Load top halo
    if (threadIdx.y < radius) {
        int topY = blockIdx.y * blockDim.y - radius + threadIdx.y;
        sharedCol[threadIdx.y] = (topY < 0) ?
            input[x] : input[topY * width + x];
    }

    // Load bottom halo
    if (threadIdx.y >= blockDim.y - radius) {
        int bottomY = blockIdx.y * blockDim.y + threadIdx.y + radius;
        int sharedBottomIdx = threadIdx.y + 2 * radius;
        sharedCol[sharedBottomIdx] = (bottomY >= height) ?
            input[(height - 1) * width + x] : input[bottomY * width + x];
    }

    __syncthreads();

    // Perform convolution
    if (y < height) {
        float sum = 0.0f;

        #pragma unroll
        for (int k = -radius; k <= radius; k++) {
            sum += sharedCol[sharedIdx + k] * kernel[k + radius];
        }

        output[y * width + x] = sum;
    }
}

/*
 * C-style wrapper functions for C++ code
 */
extern "C" {

void launchGaussianBlurHorizontal(
    const float* input,
    float* output,
    const float* kernel,
    int width,
    int height,
    int radius,
    dim3 gridSize,
    dim3 blockSize,
    int sharedMemSize
) {
    gaussianBlurHorizontalKernel<<<gridSize, blockSize, sharedMemSize>>>(
        input, output, kernel, width, height, radius
    );
}

void launchGaussianBlurVertical(
    const float* input,
    float* output,
    const float* kernel,
    int width,
    int height,
    int radius,
    dim3 gridSize,
    dim3 blockSize,
    int sharedMemSize
) {
    gaussianBlurVerticalKernel<<<gridSize, blockSize, sharedMemSize>>>(
        input, output, kernel, width, height, radius
    );
}

} // extern "C"

Writing gaussian_blur_kernels.cu


In [14]:
%%writefile main.cpp
/*
 * Main Application - C++ with CUDA Gaussian Blur
 *
 * Demonstrates usage of the GaussianBlur class
 */

#include "gaussian_blur.hpp"
#include <iostream>
#include <vector>
#include <chrono>
#include <random>
#include <iomanip>

// Simple image loader/saver simulation
class Image {
private:
    int width_;
    int height_;
    std::vector<float> data_;

public:
    Image(int width, int height)
        : width_(width), height_(height), data_(width * height) {}

    void fillRandom() {
        std::random_device rd;
        std::mt19937 gen(rd());
        std::uniform_real_distribution<float> dis(0.0f, 1.0f);

        for (auto& pixel : data_) {
            pixel = dis(gen);
        }
    }

    void fillTestPattern() {
        for (int y = 0; y < height_; y++) {
            for (int x = 0; x < width_; x++) {
                // Create a checkerboard pattern
                float value = ((x / 32) % 2) ^ ((y / 32) % 2) ? 1.0f : 0.0f;
                data_[y * width_ + x] = value;
            }
        }
    }

    float* data() { return data_.data(); }
    const float* data() const { return data_.data(); }
    int width() const { return width_; }
    int height() const { return height_; }
    size_t size() const { return data_.size(); }

    float getPixel(int x, int y) const {
        if (x < 0 || x >= width_ || y < 0 || y >= height_) {
            return 0.0f;
        }
        return data_[y * width_ + x];
    }

    void printRegion(int startX, int startY, int sizeX, int sizeY) const {
        std::cout << std::fixed << std::setprecision(3);
        for (int y = startY; y < startY + sizeY && y < height_; y++) {
            for (int x = startX; x < startX + sizeX && x < width_; x++) {
                std::cout << getPixel(x, y) << " ";
            }
            std::cout << std::endl;
        }
    }
};

// Benchmark helper
class Timer {
private:
    std::chrono::high_resolution_clock::time_point start_;

public:
    void start() {
        start_ = std::chrono::high_resolution_clock::now();
    }

    double elapsed() const {
        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> duration = end - start_;
        return duration.count();
    }
};

// Run benchmark
void runBenchmark(gpu::GaussianBlur& blur, const Image& input,
                  Image& output, int iterations = 10) {
    Timer timer;
    std::vector<double> times;

    // Warm-up run
    blur.apply(input.data(), output.data());

    // Benchmark runs
    for (int i = 0; i < iterations; i++) {
        timer.start();
        blur.apply(input.data(), output.data());
        times.push_back(timer.elapsed());
    }

    // Calculate statistics
    double sum = 0.0;
    double minTime = times[0];
    double maxTime = times[0];

    for (double t : times) {
        sum += t;
        minTime = std::min(minTime, t);
        maxTime = std::max(maxTime, t);
    }

    double avgTime = sum / iterations;
    int pixels = input.width() * input.height();
    double throughput = (pixels / 1e6) / (avgTime / 1000.0);

    std::cout << "\n=== Benchmark Results ===" << std::endl;
    std::cout << "Iterations: " << iterations << std::endl;
    std::cout << "Average time: " << avgTime << " ms" << std::endl;
    std::cout << "Min time: " << minTime << " ms" << std::endl;
    std::cout << "Max time: " << maxTime << " ms" << std::endl;
    std::cout << "Throughput: " << throughput << " Mpixels/sec" << std::endl;
}

int main(int argc, char** argv) {
    std::cout << "=== Separable Gaussian Blur (C++ with CUDA) ===" << std::endl;
    std::cout << std::endl;

    // Configuration
    const int width = 1920;
    const int height = 1080;
    const int radius = 5;
    const float sigma = 2.0f;

    std::cout << "Image dimensions: " << width << " x " << height << std::endl;
    std::cout << "Kernel radius: " << radius << std::endl;
    std::cout << "Sigma: " << sigma << std::endl;
    std::cout << std::endl;

    try {
        // Create input and output images
        Image input(width, height);
        Image output(width, height);

        // Fill with test data
        std::cout << "Generating test image..." << std::endl;
        input.fillRandom();

        // Create Gaussian blur processor
        std::cout << "Initializing GPU processor..." << std::endl;
        gpu::GaussianBlur blur(width, height, radius, sigma);

        // Print kernel
        blur.printKernel();
        std::cout << std::endl;

        // Apply blur
        std::cout << "Applying Gaussian blur..." << std::endl;
        Timer timer;
        timer.start();
        blur.apply(input.data(), output.data());
        double elapsed = timer.elapsed();

        std::cout << "First run completed in " << elapsed << " ms" << std::endl;

        // Show sample results
        std::cout << "\nSample input region (center 5x5):" << std::endl;
        input.printRegion(width/2 - 2, height/2 - 2, 5, 5);

        std::cout << "\nSample output region (center 5x5):" << std::endl;
        output.printRegion(width/2 - 2, height/2 - 2, 5, 5);

        // Run benchmark
        std::cout << "\nRunning performance benchmark..." << std::endl;
        runBenchmark(blur, input, output, 100);

        // Test with different kernel sizes
        std::cout << "\n=== Testing Different Kernel Sizes ===" << std::endl;
        std::vector<int> radii = {3, 5, 7, 10};

        for (int r : radii) {
            gpu::GaussianBlur blurTest(width, height, r, 2.0f);
            Timer t;
            t.start();
            blurTest.apply(input.data(), output.data());
            double time = t.elapsed();

            int kernelSize = 2 * r + 1;
            double throughput = (width * height / 1e6) / (time / 1000.0);

            std::cout << "Radius " << r << " (kernel " << kernelSize
                      << "x" << kernelSize << "): "
                      << time << " ms, "
                      << throughput << " Mpixels/sec" << std::endl;
        }

        std::cout << "\n=== Success! ===" << std::endl;

    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
        return 1;
    }

    return 0;
}

Writing main.cpp


In [9]:
%%writefile gaussian_blur.hpp
/*
 * Separable Gaussian Blur - C++ with CUDA Implementation
 *
 * This is a C++ wrapper around CUDA kernels providing:
 * - Object-oriented interface
 * - RAII memory management
 * - Support for multiple data types (float, uchar)
 * - Easy-to-use API
 */

#ifndef GAUSSIAN_BLUR_HPP
#define GAUSSIAN_BLUR_HPP

#include <cuda_runtime.h>
#include <vector>
#include <memory>
#include <stdexcept>
#include <cmath>
#include <iostream>

// CUDA kernel declarations
extern "C" {
    void launchGaussianBlurHorizontal(
        const float* input,
        float* output,
        const float* kernel,
        int width,
        int height,
        int radius,
        dim3 gridSize,
        dim3 blockSize,
        int sharedMemSize
    );

    void launchGaussianBlurVertical(
        const float* input,
        float* output,
        const float* kernel,
        int width,
        int height,
        int radius,
        dim3 gridSize,
        dim3 blockSize,
        int sharedMemSize
    );
}

namespace gpu {

// RAII wrapper for CUDA device memory
template<typename T>
class DeviceBuffer {
private:
    T* d_ptr_;
    size_t size_;

public:
    DeviceBuffer(size_t size) : size_(size) {
        cudaError_t err = cudaMalloc(&d_ptr_, size * sizeof(T));
        if (err != cudaSuccess) {
            throw std::runtime_error("CUDA malloc failed: " +
                std::string(cudaGetErrorString(err)));
        }
    }

    ~DeviceBuffer() {
        if (d_ptr_) {
            cudaFree(d_ptr_);
        }
    }

    // Delete copy constructor and assignment
    DeviceBuffer(const DeviceBuffer&) = delete;
    DeviceBuffer& operator=(const DeviceBuffer&) = delete;

    // Move constructor and assignment
    DeviceBuffer(DeviceBuffer&& other) noexcept
        : d_ptr_(other.d_ptr_), size_(other.size_) {
        other.d_ptr_ = nullptr;
    }

    DeviceBuffer& operator=(DeviceBuffer&& other) noexcept {
        if (this != &other) {
            if (d_ptr_) cudaFree(d_ptr_);
            d_ptr_ = other.d_ptr_;
            size_ = other.size_;
            other.d_ptr_ = nullptr;
        }
        return *this;
    }

    T* get() { return d_ptr_; }
    const T* get() const { return d_ptr_; }
    size_t size() const { return size_; }

    void copyToDevice(const T* host_data) {
        cudaError_t err = cudaMemcpy(d_ptr_, host_data,
            size_ * sizeof(T), cudaMemcpyHostToDevice);
        if (err != cudaSuccess) {
            throw std::runtime_error("Copy to device failed");
        }
    }

    void copyToHost(T* host_data) const {
        cudaError_t err = cudaMemcpy(host_data, d_ptr_,
            size_ * sizeof(T), cudaMemcpyDeviceToHost);
        if (err != cudaSuccess) {
            throw std::runtime_error("Copy to host failed");
        }
    }
};

// Gaussian blur class
class GaussianBlur {
private:
    int width_;
    int height_;
    int radius_;
    float sigma_;
    std::vector<float> kernel_;

    std::unique_ptr<DeviceBuffer<float>> d_kernel_;
    std::unique_ptr<DeviceBuffer<float>> d_temp_;

    void generateKernel() {
        int kernelSize = 2 * radius_ + 1;
        kernel_.resize(kernelSize);

        float sum = 0.0f;
        for (int i = -radius_; i <= radius_; i++) {
            float value = std::exp(-(i * i) / (2.0f * sigma_ * sigma_));
            kernel_[i + radius_] = value;
            sum += value;
        }

        // Normalize
        for (auto& val : kernel_) {
            val /= sum;
        }
    }

public:
    GaussianBlur(int width, int height, int radius, float sigma)
        : width_(width), height_(height), radius_(radius), sigma_(sigma) {

        if (radius <= 0 || radius > 50) {
            throw std::invalid_argument("Radius must be between 1 and 50");
        }

        if (sigma <= 0.0f) {
            throw std::invalid_argument("Sigma must be positive");
        }

        // Generate Gaussian kernel
        generateKernel();

        // Allocate device memory
        d_kernel_ = std::make_unique<DeviceBuffer<float>>(kernel_.size());
        d_kernel_->copyToDevice(kernel_.data());

        // Allocate temporary buffer for intermediate results
        d_temp_ = std::make_unique<DeviceBuffer<float>>(width * height);
    }

    // Apply Gaussian blur
    void apply(const float* h_input, float* h_output) {
        // Allocate device buffers
        DeviceBuffer<float> d_input(width_ * height_);
        DeviceBuffer<float> d_output(width_ * height_);

        // Copy input to device
        d_input.copyToDevice(h_input);

        // Configure kernel launch parameters
        const int blockSize = 16;
        dim3 blockDim(blockSize, blockSize);
        dim3 gridDim(
            (width_ + blockSize - 1) / blockSize,
            (height_ + blockSize - 1) / blockSize
        );

        int sharedMemSize = (blockSize + 2 * radius_) * sizeof(float);

        // Launch horizontal blur
        launchGaussianBlurHorizontal(
            d_input.get(),
            d_temp_->get(),
            d_kernel_->get(),
            width_,
            height_,
            radius_,
            gridDim,
            blockDim,
            sharedMemSize
        );

        // Check for kernel launch errors
        cudaError_t err = cudaGetLastError();
        if (err != cudaSuccess) {
            throw std::runtime_error("Horizontal kernel launch failed: " +
                std::string(cudaGetErrorString(err)));
        }

        // Launch vertical blur
        launchGaussianBlurVertical(
            d_temp_->get(),
            d_output.get(),
            d_kernel_->get(),
            width_,
            height_,
            radius_,
            gridDim,
            blockDim,
            sharedMemSize
        );

        err = cudaGetLastError();
        if (err != cudaSuccess) {
            throw std::runtime_error("Vertical kernel launch failed: " +
                std::string(cudaGetErrorString(err)));
        }

        // Synchronize
        cudaDeviceSynchronize();

        // Copy result back to host
        d_output.copyToHost(h_output);
    }

    // Getters
    int getWidth() const { return width_; }
    int getHeight() const { return height_; }
    int getRadius() const { return radius_; }
    float getSigma() const { return sigma_; }
    const std::vector<float>& getKernel() const { return kernel_; }

    // Print kernel coefficients
    void printKernel() const {
        std::cout << "Gaussian kernel (radius=" << radius_
                  << ", sigma=" << sigma_ << "):\n";
        for (size_t i = 0; i < kernel_.size(); i++) {
            std::cout << kernel_[i] << " ";
        }
        std::cout << std::endl;
    }
};

} // namespace gpu

#endif // GAUSSIAN_BLUR_HPP

Overwriting gaussian_blur.hpp


In [11]:
%%writefile Makefile
# Makefile for Gaussian Blur CUDA implementations
# Supports both standalone CUDA and C++ with CUDA versions

# Compiler settings
NVCC = nvcc
CXX = g++

# CUDA architecture (adjust based on your GPU)
# Common options:
# - sm_50: Maxwell (GTX 900 series)
# - sm_60: Pascal (GTX 10 series)
# - sm_70: Volta (Tesla V100)
# - sm_75: Turing (RTX 20 series)
# - sm_80: Ampere (RTX 30 series, A100)
# - sm_86: Ampere (RTX 30 series mobile)
# - sm_89: Ada Lovelace (RTX 40 series)
CUDA_ARCH = sm_75

# Compiler flags
NVCC_FLAGS = -arch=$(CUDA_ARCH) -O3 --use_fast_math -Xcompiler -Wall
CXX_FLAGS = -std=c++11 -Wall -O3

# CUDA include and library paths (adjust if needed)
CUDA_INC = /usr/local/cuda/include
CUDA_LIB = /usr/local/cuda/lib64

# Targets
TARGETS = gaussian_blur_cuda gaussian_blur_cpp

.PHONY: all clean

all: $(TARGETS)

# Standalone CUDA implementation
gaussian_blur_cuda: gaussian_blur_cuda.cu
	$(NVCC) $(NVCC_FLAGS) $< -o $@
	@echo "Built standalone CUDA implementation: $@"

# C++ with CUDA implementation (requires separate compilation)
gaussian_blur_kernels.o: gaussian_blur_kernels.cu
	$(NVCC) $(NVCC_FLAGS) -c $< -o $@

main.o: main.cpp gaussian_blur.hpp
	$(NVCC) $(NVCC_FLAGS) -x cu -c $< -o $@

gaussian_blur_cpp: main.o gaussian_blur_kernels.o
	$(NVCC) $(NVCC_FLAGS) $^ -o $@
	@echo "Built C++ with CUDA implementation: $@"

# Run targets
run_cuda: gaussian_blur_cuda
	@echo "\n=== Running Standalone CUDA Implementation ==="
	./gaussian_blur_cuda

run_cpp: gaussian_blur_cpp
	@echo "\n=== Running C++ with CUDA Implementation ==="
	./gaussian_blur_cpp

run: run_cuda run_cpp

# Clean build artifacts
clean:
	rm -f $(TARGETS) *.o
	@echo "Cleaned build artifacts"

# Help target
help:
	@echo "Gaussian Blur CUDA Makefile"
	@echo ""
	@echo "Targets:"
	@echo "  all              - Build all implementations (default)"
	@echo "  gaussian_blur_cuda - Build standalone CUDA version"
	@echo "  gaussian_blur_cpp  - Build C++ with CUDA version"
	@echo "  run_cuda         - Build and run standalone CUDA"
	@echo "  run_cpp          - Build and run C++ with CUDA"
	@echo "  run              - Run both implementations"
	@echo "  clean            - Remove build artifacts"
	@echo "  help             - Show this help message"
	@echo ""
	@echo "Configuration:"
	@echo "  CUDA_ARCH=$(CUDA_ARCH) - Target GPU architecture"
	@echo ""
	@echo "Usage examples:"
	@echo "  make                    # Build all"
	@echo "  make run                # Build and run both"
	@echo "  make CUDA_ARCH=sm_80    # Build for Ampere (RTX 30 series)"
	@echo "  make clean              # Clean build files"

Overwriting Makefile


In [15]:
!make run


=== Running Standalone CUDA Implementation ===
./gaussian_blur_cuda
Separable Gaussian Blur (CUDA)
Image size: 1920 x 1080
Kernel radius: 5 (size: 11)
Sigma: 2.00

Gaussian kernel coefficients:
0.008812 0.027144 0.065114 0.121649 0.176998 0.200565 0.176998 0.121649 0.065114 0.027144 0.008812 

Execution time: 0.365 ms
Throughput: 5688.70 Mpixels/sec

Sample output values:
Pixel[540][960]: 0.545790
Pixel[540][961]: 0.545790
Pixel[540][962]: 0.545790
Pixel[540][963]: 0.545790
Pixel[540][964]: 0.545790

Gaussian blur completed successfully!
nvcc -arch=sm_75 -O3 --use_fast_math -Xcompiler -Wall -x cu -c main.cpp -o main.o
nvcc -arch=sm_75 -O3 --use_fast_math -Xcompiler -Wall -c gaussian_blur_kernels.cu -o gaussian_blur_kernels.o
nvcc -arch=sm_75 -O3 --use_fast_math -Xcompiler -Wall main.o gaussian_blur_kernels.o -o gaussian_blur_cpp
Built C++ with CUDA implementation: gaussian_blur_cpp

=== Running C++ with CUDA Implementation ===
./gaussian_blur_cpp
=== Separable Gaussian Blur (C++ with 

In [17]:
%%writefile run_comparison.sh
#!/bin/bash
# Simple script to build and run Gaussian Blur comparison

set -e

echo "========================================"
echo "Gaussian Blur Comparison Runner"
echo "========================================"
echo ""

# Check if CUDA is available
if command -v nvcc &> /dev/null; then
    CUDA_AVAILABLE=true
    echo "✓ CUDA compiler found: $(nvcc --version | grep release)"
else
    CUDA_AVAILABLE=false
    echo "✗ CUDA compiler not found (nvcc not in PATH)"
fi

# Check if Python is available
if command -v python3 &> /dev/null; then
    PYTHON_AVAILABLE=true
    echo "✓ Python found: $(python3 --version)"

    # Check for numpy and scipy
    if python3 -c "import numpy, scipy" &> /dev/null; then
        echo "✓ NumPy and SciPy installed"
    else
        echo "✗ NumPy or SciPy not installed"
        echo "  Install with: pip install numpy scipy"
        PYTHON_AVAILABLE=false
    fi
else
    PYTHON_AVAILABLE=false
    echo "✗ Python3 not found"
fi

echo ""

# Default parameters
WIDTH=1000
HEIGHT=1000
KERNEL_SIZE=11
SIGMA=2.0
PATTERN="random"
ITERATIONS=10

# Parse arguments
while [[ $# -gt 0 ]]; do
    case $1 in
        --cuda-only)
            PYTHON_AVAILABLE=false
            shift
            ;;
        --python-only)
            CUDA_AVAILABLE=false
            shift
            ;;
        --width)
            WIDTH="$2"
            shift 2
            ;;
        --height)
            HEIGHT="$2"
            shift 2
            ;;
        --kernel-size)
            KERNEL_SIZE="$2"
            shift 2
            ;;
        --sigma)
            SIGMA="$2"
            shift 2
            ;;
        --pattern)
            PATTERN="$2"
            shift 2
            ;;
        --iterations)
            ITERATIONS="$2"
            shift 2
            ;;
        --help)
            echo "Usage: $0 [OPTIONS]"
            echo ""
            echo "Options:"
            echo "  --cuda-only          Run only CUDA version"
            echo "  --python-only        Run only Python version"
            echo "  --width N            Image width (default: 1000)"
            echo "  --height N           Image height (default: 1000)"
            echo "  --kernel-size N      Kernel size (default: 11)"
            echo "  --sigma N            Gaussian sigma (default: 2.0)"
            echo "  --pattern TYPE       Pattern type (default: random)"
            echo "  --iterations N       Benchmark iterations (default: 10)"
            echo "  --help               Show this help"
            echo ""
            echo "Examples:"
            echo "  $0                                    # Run both versions"
            echo "  $0 --cuda-only --width 1920 --height 1080"
            echo "  $0 --python-only --kernel-size 15"
            exit 0
            ;;
        *)
            echo "Unknown option: $1"
            echo "Use --help for usage information"
            exit 1
            ;;
    esac
done

PARAMS="--width $WIDTH --height $HEIGHT --kernel-size $KERNEL_SIZE --sigma $SIGMA --pattern $PATTERN --iterations $ITERATIONS"

echo "Configuration:"
echo "  Image: ${WIDTH}x${HEIGHT}"
echo "  Kernel: ${KERNEL_SIZE}x${KERNEL_SIZE}"
echo "  Sigma: $SIGMA"
echo "  Pattern: $PATTERN"
echo "  Iterations: $ITERATIONS"
echo ""

# Run CUDA version
if [ "$CUDA_AVAILABLE" = true ]; then
    echo "========================================"
    echo "Building and Running CUDA Version"
    echo "========================================"
    echo ""

    if [ ! -f "gaussian_blur_comparison_cuda" ]; then
        echo "Building CUDA application..."
        make -f Makefile_comparison cuda
        echo ""
    fi

    echo "Running CUDA version..."
    ./gaussian_blur_comparison_cuda $PARAMS
    echo ""
fi

# Run Python version
if [ "$PYTHON_AVAILABLE" = true ]; then
    echo "========================================"
    echo "Running Python Version"
    echo "========================================"
    echo ""

    python3 gaussian_blur_comparison.py $PARAMS
    echo ""
fi

# Summary
echo "========================================"
echo "Execution Summary"
echo "========================================"

if [ "$CUDA_AVAILABLE" = true ]; then
    echo "✓ CUDA version executed successfully"
else
    echo "✗ CUDA version not executed (not available)"
fi

if [ "$PYTHON_AVAILABLE" = true ]; then
    echo "✓ Python version executed successfully"
else
    echo "✗ Python version not executed (not available)"
fi

if [ "$CUDA_AVAILABLE" = true ] && [ "$PYTHON_AVAILABLE" = true ]; then
    echo ""
    echo "Both versions completed! Compare the results above."
fi

echo ""

Overwriting run_comparison.sh


In [20]:
%%writefile gaussian_blur_comparison.py
#!/usr/bin/env python3
"""
Gaussian Blur: Complete Comparison
Separable Convolution vs Full 2D Convolution

This standalone application demonstrates both approaches and verifies they produce
identical results while showing the performance difference.
"""

import numpy as np
from scipy import ndimage
import time
import argparse


def gaussian_2d_kernel(sigma=1.0, kernel_size=5):
    """
    Create 2D Gaussian kernel

    Args:
        sigma: Standard deviation of Gaussian
        kernel_size: Size of kernel (must be odd)

    Returns:
        2D numpy array of shape (kernel_size, kernel_size)
    """
    if kernel_size % 2 == 0:
        kernel_size += 1  # Ensure odd size

    x = np.arange(kernel_size) - kernel_size // 2
    y = np.arange(kernel_size) - kernel_size // 2
    x, y = np.meshgrid(x, y)

    # 2D Gaussian formula: G(x,y) = (1/(2πσ²)) * exp(-(x²+y²)/(2σ²))
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()  # Normalize

    return kernel


def gaussian_1d_kernel(sigma=1.0, kernel_size=5):
    """
    Create 1D Gaussian kernel

    Args:
        sigma: Standard deviation of Gaussian
        kernel_size: Size of kernel (must be odd)

    Returns:
        1D numpy array of shape (kernel_size,)
    """
    if kernel_size % 2 == 0:
        kernel_size += 1

    x = np.arange(kernel_size) - kernel_size // 2

    # 1D Gaussian formula: G(x) = (1/√(2πσ²)) * exp(-x²/(2σ²))
    kernel = np.exp(-(x**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()  # Normalize

    return kernel


def gaussian_2d_full(image, sigma=1.0, kernel_size=5):
    """
    Apply Gaussian blur using full 2D convolution

    Args:
        image: Input image (2D numpy array)
        sigma: Standard deviation
        kernel_size: Size of kernel

    Returns:
        Blurred image
    """
    kernel = gaussian_2d_kernel(sigma, kernel_size)
    return ndimage.convolve(image, kernel, mode="reflect")


def gaussian_separable(image, sigma=1.0, kernel_size=5):
    """
    Apply Gaussian blur using separable convolution (two 1D passes)

    Args:
        image: Input image (2D numpy array)
        sigma: Standard deviation
        kernel_size: Size of kernel

    Returns:
        Blurred image
    """
    kernel_1d = gaussian_1d_kernel(sigma, kernel_size)

    # First pass: convolve along rows (horizontal)
    temp = ndimage.convolve1d(image, kernel_1d, axis=1, mode="reflect")

    # Second pass: convolve along columns (vertical)
    result = ndimage.convolve1d(temp, kernel_1d, axis=0, mode="reflect")

    return result


def create_test_image(width=1000, height=1000, pattern="gradient"):
    """
    Create a test image with various patterns

    Args:
        width: Image width
        height: Image height
        pattern: Type of pattern ('random', 'gradient', 'checkerboard', 'gaussian')

    Returns:
        2D numpy array
    """
    if pattern == "random":
        return np.random.rand(height, width).astype(np.float32)

    elif pattern == "gradient":
        x = np.linspace(0, 1, width)
        y = np.linspace(0, 1, height)
        X, Y = np.meshgrid(x, y)
        return (X + Y) / 2

    elif pattern == "checkerboard":
        image = np.zeros((height, width), dtype=np.float32)
        square_size = 32
        for i in range(0, height, square_size):
            for j in range(0, width, square_size):
                if ((i // square_size) + (j // square_size)) % 2 == 0:
                    image[i : i + square_size, j : j + square_size] = 1.0
        return image

    elif pattern == "gaussian":
        # Create a Gaussian blob in the center
        x = np.arange(width) - width // 2
        y = np.arange(height) - height // 2
        X, Y = np.meshgrid(x, y)
        return np.exp(-(X**2 + Y**2) / (2 * (min(width, height) / 6) ** 2))

    else:
        raise ValueError(f"Unknown pattern: {pattern}")


def benchmark_method(method_func, image, sigma, kernel_size, iterations=10):
    """
    Benchmark a blur method

    Args:
        method_func: Function to benchmark
        image: Input image
        sigma: Gaussian sigma
        kernel_size: Kernel size
        iterations: Number of iterations

    Returns:
        Dictionary with timing statistics
    """
    times = []

    # Warm-up run
    result = method_func(image, sigma, kernel_size)

    # Benchmark runs
    for _ in range(iterations):
        start = time.time()
        result = method_func(image, sigma, kernel_size)
        end = time.time()
        times.append((end - start) * 1000)  # Convert to ms

    return {
        "result": result,
        "min": min(times),
        "max": max(times),
        "mean": np.mean(times),
        "std": np.std(times),
        "median": np.median(times),
    }


def verify_results(result1, result2, tolerance=1e-10):
    """
    Verify that two results are identical within tolerance

    Args:
        result1: First result
        result2: Second result
        tolerance: Maximum acceptable difference

    Returns:
        Dictionary with verification statistics
    """
    diff = np.abs(result1 - result2)

    return {
        "max_diff": np.max(diff),
        "mean_diff": np.mean(diff),
        "identical": np.max(diff) < tolerance,
        "tolerance": tolerance,
    }


def print_kernel_info(sigma, kernel_size):
    """Print kernel information"""
    kernel_1d = gaussian_1d_kernel(sigma, kernel_size)
    kernel_2d = gaussian_2d_kernel(sigma, kernel_size)

    print(f"\n{'=' * 70}")
    print(f"Kernel Information (σ={sigma}, size={kernel_size}x{kernel_size})")
    print(f"{'=' * 70}")

    print(f"\n1D Kernel ({kernel_size} elements):")
    print(f"  Values: {kernel_1d}")
    print(f"  Sum: {kernel_1d.sum():.10f} (should be 1.0)")

    print(f"\n2D Kernel ({kernel_size}x{kernel_size} elements):")
    print(f"  Center row: {kernel_2d[kernel_size // 2, :]}")
    print(f"  Sum: {kernel_2d.sum():.10f} (should be 1.0)")

    # Verify separability
    outer_product = np.outer(kernel_1d, kernel_1d)
    separable_diff = np.max(np.abs(kernel_2d - outer_product))
    print(f"\n  Separability check:")
    print(f"    Max difference between 2D and outer(1D,1D): {separable_diff:.2e}")
    print(f"    Kernel is separable: {separable_diff < 1e-10}")


def print_benchmark_results(name, stats, image_size, kernel_size):
    """Print benchmark statistics"""
    height, width = image_size
    pixels = width * height
    throughput = (pixels / 1e6) / (stats["mean"] / 1000.0)
    ops_per_pixel = kernel_size * kernel_size if "2D" in name else 2 * kernel_size

    print(f"\n{name}:")
    print(f"  Time: {stats['mean']:.3f} ± {stats['std']:.3f} ms")
    print(f"  Range: [{stats['min']:.3f}, {stats['max']:.3f}] ms")
    print(f"  Throughput: {throughput:.2f} Mpixels/sec")
    print(f"  Operations/pixel: {ops_per_pixel}")


def save_output(filename, image):
    """Save image to file (as numpy array)"""
    np.save(filename, image)
    print(f"\nSaved output to: {filename}")


def main():
    parser = argparse.ArgumentParser(
        description="Gaussian Blur Comparison: Separable vs Full 2D Convolution"
    )
    parser.add_argument(
        "--width", type=int, default=1000, help="Image width (default: 1000)"
    )
    parser.add_argument(
        "--height", type=int, default=1000, help="Image height (default: 1000)"
    )
    parser.add_argument(
        "--sigma", type=float, default=2.0, help="Gaussian sigma (default: 2.0)"
    )
    parser.add_argument(
        "--kernel-size", type=int, default=11, help="Kernel size (default: 11)"
    )
    parser.add_argument(
        "--pattern",
        type=str,
        default="random",
        choices=["random", "gradient", "checkerboard", "gaussian"],
        help="Test pattern (default: random)",
    )
    parser.add_argument(
        "--iterations", type=int, default=10, help="Benchmark iterations (default: 10)"
    )
    parser.add_argument("--save", action="store_true", help="Save output images")

    args = parser.parse_args([])

    # Header
    print("=" * 70)
    print("Gaussian Blur: Separable vs Full 2D Convolution")
    print("=" * 70)

    # Configuration
    print(f"\nConfiguration:")
    print(f"  Image size: {args.width} x {args.height}")
    print(f"  Kernel size: {args.kernel_size} x {args.kernel_size}")
    print(f"  Sigma: {args.sigma}")
    print(f"  Pattern: {args.pattern}")
    print(f"  Benchmark iterations: {args.iterations}")

    # Create test image
    print(f"\nCreating test image ({args.pattern} pattern)...")
    image = create_test_image(args.width, args.height, args.pattern)

    # Print kernel information
    print_kernel_info(args.sigma, args.kernel_size)

    # Benchmark Full 2D Convolution
    print(f"\n{'=' * 70}")
    print("Benchmarking Full 2D Convolution")
    print(f"{'=' * 70}")
    stats_2d = benchmark_method(
        gaussian_2d_full, image, args.sigma, args.kernel_size, args.iterations
    )
    print_benchmark_results(
        "Full 2D Convolution", stats_2d, (args.height, args.width), args.kernel_size
    )

    # Benchmark Separable Convolution
    print(f"\n{'=' * 70}")
    print("Benchmarking Separable Convolution")
    print(f"{'=' * 70}")
    stats_sep = benchmark_method(
        gaussian_separable, image, args.sigma, args.kernel_size, args.iterations
    )
    print_benchmark_results(
        "Separable Convolution", stats_sep, (args.height, args.width), args.kernel_size
    )

    # Verify results are identical
    print(f"\n{'=' * 70}")
    print("Verification")
    print(f"{'=' * 70}")
    verification = verify_results(stats_2d["result"], stats_sep["result"])

    print(f"\nResult comparison:")
    print(f"  Max difference: {verification['max_diff']:.2e}")
    print(f"  Mean difference: {verification['mean_diff']:.2e}")
    print(
        f"  Results identical (tolerance={verification['tolerance']:.2e}): "
        f"{verification['identical']}"
    )

    # Performance comparison
    print(f"\n{'=' * 70}")
    print("Performance Summary")
    print(f"{'=' * 70}")

    speedup = stats_2d["mean"] / stats_sep["mean"]
    theoretical_speedup = args.kernel_size / 2

    print(f"\nSpeedup (Separable vs Full 2D):")
    print(f"  Actual: {speedup:.2f}x")
    print(f"  Theoretical: {theoretical_speedup:.2f}x")
    print(f"  Efficiency: {(speedup / theoretical_speedup) * 100:.1f}%")

    # Sample output values
    print(f"\n{'=' * 70}")
    print("Sample Output Values (center 5x5 region)")
    print(f"{'=' * 70}")

    cy, cx = args.height // 2, args.width // 2
    print("\nInput:")
    print(image[cy - 2 : cy + 3, cx - 2 : cx + 3])

    print("\nOutput (both methods produce identical results):")
    print(stats_sep["result"][cy - 2 : cy + 3, cx - 2 : cx + 3])

    # Save outputs if requested
    if args.save:
        save_output("input.npy", image)
        save_output("output_2d.npy", stats_2d["result"])
        save_output("output_separable.npy", stats_sep["result"])

    # Test multiple kernel sizes
    print(f"\n{'=' * 70}")
    print("Performance Scaling with Kernel Size")
    print(f"{'=' * 70}")

    print(
        f"\n{'Kernel':<10} {'Size':<8} {'2D Time':<12} {'Sep Time':<12} {'Speedup':<10} {'Theoretical'}"
    )
    print("-" * 70)

    for ksize in [3, 5, 7, 9, 11, 15, 21]:
        if ksize > args.kernel_size:
            break

        # Quick benchmark (fewer iterations)
        stats_2d_test = benchmark_method(
            gaussian_2d_full, image, args.sigma, ksize, iterations=3
        )
        stats_sep_test = benchmark_method(
            gaussian_separable, image, args.sigma, ksize, iterations=3
        )

        speedup_test = stats_2d_test["mean"] / stats_sep_test["mean"]
        theoretical = ksize / 2

        print(
            f"{ksize}x{ksize:<6} {ksize * ksize:<8} "
            f"{stats_2d_test['mean']:>10.2f}ms "
            f"{stats_sep_test['mean']:>10.2f}ms "
            f"{speedup_test:>8.2f}x "
            f"{theoretical:>10.2f}x"
        )

    print(f"\n{'=' * 70}")
    print("Completed Successfully!")
    print(f"{'=' * 70}\n")


if __name__ == "__main__":
    main()


Writing gaussian_blur_comparison.py


In [22]:
%%writefile Makefile_comparison
# Makefile for Gaussian Blur Comparison
# Builds both Python and C++/CUDA standalone applications

NVCC = nvcc
PYTHON = python3

# CUDA architecture
CUDA_ARCH = sm_75

# Compiler flags
NVCC_FLAGS = -arch=$(CUDA_ARCH) -O3 --use_fast_math -Xcompiler -Wall -std=c++11

# Targets
CUDA_TARGET = gaussian_blur_comparison_cuda
PYTHON_SCRIPT = gaussian_blur_comparison.py

.PHONY: all cuda python run_cuda run_python run compare clean help

all: cuda

# Build CUDA version
cuda: $(CUDA_TARGET)

$(CUDA_TARGET): gaussian_blur_comparison_cuda.cu
	$(NVCC) $(NVCC_FLAGS) $< -o $@
	@echo "Built CUDA comparison application: $@"

# Python doesn't need compilation, but we can check dependencies
python:
	@echo "Checking Python dependencies..."
	@$(PYTHON) -c "import numpy; import scipy; print('✓ NumPy and SciPy are installed')" || \
		(echo "✗ Missing dependencies. Install with: pip install numpy scipy" && exit 1)
	@echo "Python script is ready: $(PYTHON_SCRIPT)"

# Run CUDA version with default parameters
run_cuda: $(CUDA_TARGET)
	@echo ""
	@echo "========================================"
	@echo "Running CUDA Comparison"
	@echo "========================================"
	@./$(CUDA_TARGET)

# Run Python version with default parameters
run_python: python
	@echo ""
	@echo "========================================"
	@echo "Running Python Comparison"
	@echo "========================================"
	@$(PYTHON) $(PYTHON_SCRIPT)

# Run both and compare
run: run_cuda run_python

# Compare with custom parameters
compare: $(CUDA_TARGET) python
	@echo ""
	@echo "========================================"
	@echo "Comparison: CUDA vs Python"
	@echo "========================================"
	@echo ""
	@echo "--- CUDA Version ---"
	@./$(CUDA_TARGET) --width 500 --height 500 --kernel-size 11 --pattern gradient
	@echo ""
	@echo "--- Python Version ---"
	@$(PYTHON) $(PYTHON_SCRIPT) --width 500 --height 500 --kernel-size 11 --pattern gradient

# Example runs with different configurations
test_small: $(CUDA_TARGET) python
	@echo "Testing with small image (256x256)..."
	./$(CUDA_TARGET) --width 256 --height 256 --kernel-size 7

test_large: $(CUDA_TARGET) python
	@echo "Testing with large image (2048x2048)..."
	./$(CUDA_TARGET) --width 2048 --height 2048 --kernel-size 11

test_patterns: $(CUDA_TARGET)
	@echo "Testing different patterns..."
	@echo "\n=== Random ==="
	@./$(CUDA_TARGET) --width 500 --height 500 --pattern random --iterations 5
	@echo "\n=== Gradient ==="
	@./$(CUDA_TARGET) --width 500 --height 500 --pattern gradient --iterations 5
	@echo "\n=== Checkerboard ==="
	@./$(CUDA_TARGET) --width 500 --height 500 --pattern checkerboard --iterations 5

# Benchmark different kernel sizes
benchmark: $(CUDA_TARGET)
	@echo "Benchmarking different kernel sizes..."
	@for size in 3 5 7 9 11 15 21; do \
		echo "\n=== Kernel size: $$size ==="; \
		./$(CUDA_TARGET) --kernel-size $$size --iterations 20; \
	done

# Clean build artifacts
clean:
	rm -f $(CUDA_TARGET) *.o
	rm -f *.npy  # Clean up any saved numpy arrays
	@echo "Cleaned build artifacts"

# Help
help:
	@echo "Gaussian Blur Comparison Makefile"
	@echo ""
	@echo "Targets:"
	@echo "  all              - Build CUDA application (default)"
	@echo "  cuda             - Build CUDA application"
	@echo "  python           - Check Python dependencies"
	@echo "  run_cuda         - Build and run CUDA version"
	@echo "  run_python       - Run Python version"
	@echo "  run              - Run both versions"
	@echo "  compare          - Run both with same parameters"
	@echo "  test_small       - Test with small image (256x256)"
	@echo "  test_large       - Test with large image (2048x2048)"
	@echo "  test_patterns    - Test all image patterns"
	@echo "  benchmark        - Benchmark different kernel sizes"
	@echo "  clean            - Remove build artifacts"
	@echo "  help             - Show this help"
	@echo ""
	@echo "Command-line arguments (for manual runs):"
	@echo "  --width N          Image width (default: 1000)"
	@echo "  --height N         Image height (default: 1000)"
	@echo "  --sigma N          Gaussian sigma (default: 2.0)"
	@echo "  --kernel-size N    Kernel size (default: 11)"
	@echo "  --pattern TYPE     Pattern: random, gradient, checkerboard (default: random)"
	@echo "  --iterations N     Benchmark iterations (default: 10)"
	@echo ""
	@echo "Examples:"
	@echo "  make                                    # Build CUDA version"
	@echo "  make run_cuda                           # Run CUDA version"
	@echo "  make run_python                         # Run Python version"
	@echo "  make compare                            # Compare both"
	@echo "  make benchmark                          # Benchmark kernel sizes"
	@echo "  ./$(CUDA_TARGET) --width 1920 --height 1080 --kernel-size 15"
	@echo "  $(PYTHON) $(PYTHON_SCRIPT) --width 1920 --height 1080 --kernel-size 15"


Writing Makefile_comparison


In [27]:
%%writefile gaussian_blur_comparison_cuda.cu
/*
 * Gaussian Blur Comparison - Complete C++/CUDA Implementation
 *
 * Standalone application comparing:
 * 1. Full 2D convolution
 * 2. Separable convolution (2x 1D passes)
 *
 * Produces identical output to the Python version for verification
 */

#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <string.h>

#define BLOCK_SIZE 16
#define MAX_KERNEL_RADIUS 50
#define MAX_KERNEL_SIZE (2 * MAX_KERNEL_RADIUS + 1)

// Error checking macro
#define CUDA_CHECK(call) \
    do { \
        cudaError_t error = call; \
        if (error != cudaSuccess) { \
            fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, \
                    cudaGetErrorString(error)); \
            exit(EXIT_FAILURE); \
        } \
    } while(0)

// ============================================================================
// Kernel Generation Functions
// ============================================================================

void generateGaussianKernel1D(float* kernel, int radius, float sigma) {
    float sum = 0.0f;
    int size = 2 * radius + 1;

    for (int i = 0; i < size; i++) {
        int x = i - radius;
        float value = expf(-(x * x) / (2.0f * sigma * sigma));
        kernel[i] = value;
        sum += value;
    }

    // Normalize
    for (int i = 0; i < size; i++) {
        kernel[i] /= sum;
    }
}

void generateGaussianKernel2D(float* kernel, int radius, float sigma) {
    float sum = 0.0f;
    int size = 2 * radius + 1;

    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            int x = i - radius;
            int y = j - radius;
            float value = expf(-(x * x + y * y) / (2.0f * sigma * sigma));
            kernel[i * size + j] = value;
            sum += value;
        }
    }

    // Normalize
    for (int i = 0; i < size * size; i++) {
        kernel[i] /= sum;
    }
}

// ============================================================================
// CUDA Kernels - Full 2D Convolution
// ============================================================================

__global__ void gaussianBlur2DKernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    const float* __restrict__ kernel,
    int width,
    int height,
    int radius
) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width || y >= height) return;

    int kernelSize = 2 * radius + 1;
    float sum = 0.0f;

    // 2D convolution
    for (int ky = -radius; ky <= radius; ky++) {
        for (int kx = -radius; kx <= radius; kx++) {
            // Reflect boundary conditions
            int ix = x + kx;
            int iy = y + ky;

            // Clamp to edges
            ix = max(0, min(width - 1, ix));
            iy = max(0, min(height - 1, iy));

            int kernelIdx = (ky + radius) * kernelSize + (kx + radius);
            sum += input[iy * width + ix] * kernel[kernelIdx];
        }
    }

    output[y * width + x] = sum;
}

// ============================================================================
// CUDA Kernels - Separable Convolution
// ============================================================================

__global__ void gaussianBlurHorizontalKernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    const float* __restrict__ kernel,
    int width,
    int height,
    int radius
) {
    extern __shared__ float sharedRow[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (y >= height) return;

    int sharedIdx = threadIdx.x + radius;

    // Load center data
    if (x < width) {
        sharedRow[sharedIdx] = input[y * width + x];
    }

    // Load left halo
    if (threadIdx.x < radius) {
        int leftX = blockIdx.x * blockDim.x - radius + threadIdx.x;
        if (leftX < 0) leftX = 0;
        sharedRow[threadIdx.x] = input[y * width + leftX];
    }

    // Load right halo
    if (threadIdx.x >= blockDim.x - radius) {
        int rightX = blockIdx.x * blockDim.x + threadIdx.x + radius;
        int sharedRightIdx = threadIdx.x + 2 * radius;
        if (rightX >= width) rightX = width - 1;
        sharedRow[sharedRightIdx] = input[y * width + rightX];
    }

    __syncthreads();

    // Perform convolution
    if (x < width) {
        float sum = 0.0f;
        for (int k = -radius; k <= radius; k++) {
            sum += sharedRow[sharedIdx + k] * kernel[k + radius];
        }
        output[y * width + x] = sum;
    }
}

__global__ void gaussianBlurVerticalKernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    const float* __restrict__ kernel,
    int width,
    int height,
    int radius
) {
    extern __shared__ float sharedCol[];

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= width) return;

    int sharedIdx = threadIdx.y + radius;

    // Load center data
    if (y < height) {
        sharedCol[sharedIdx] = input[y * width + x];
    }

    // Load top halo
    if (threadIdx.y < radius) {
        int topY = blockIdx.y * blockDim.y - radius + threadIdx.y;
        if (topY < 0) topY = 0;
        sharedCol[threadIdx.y] = input[topY * width + x];
    }

    // Load bottom halo
    if (threadIdx.y >= blockDim.y - radius) {
        int bottomY = blockIdx.y * blockDim.y + threadIdx.y + radius;
        int sharedBottomIdx = threadIdx.y + 2 * radius;
        if (bottomY >= height) bottomY = height - 1;
        sharedCol[sharedBottomIdx] = input[bottomY * width + x];
    }

    __syncthreads();

    // Perform convolution
    if (y < height) {
        float sum = 0.0f;
        for (int k = -radius; k <= radius; k++) {
            sum += sharedCol[sharedIdx + k] * kernel[k + radius];
        }
        output[y * width + x] = sum;
    }
}

// ============================================================================
// Host Functions
// ============================================================================

void createTestImage(float* image, int width, int height, const char* pattern) {
    if (strcmp(pattern, "random") == 0) {
        srand(42);  // Fixed seed for reproducibility
        for (int i = 0; i < width * height; i++) {
            image[i] = (float)rand() / RAND_MAX;
        }
    }
    else if (strcmp(pattern, "gradient") == 0) {
        for (int y = 0; y < height; y++) {
            for (int x = 0; x < width; x++) {
                float fx = (float)x / (width - 1);
                float fy = (float)y / (height - 1);
                image[y * width + x] = (fx + fy) / 2.0f;
            }
        }
    }
    else if (strcmp(pattern, "checkerboard") == 0) {
        int squareSize = 32;
        for (int y = 0; y < height; y++) {
            for (int x = 0; x < width; x++) {
                int bx = x / squareSize;
                int by = y / squareSize;
                image[y * width + x] = ((bx + by) % 2 == 0) ? 1.0f : 0.0f;
            }
        }
    }
    else {
        fprintf(stderr, "Unknown pattern: %s\n", pattern);
        exit(1);
    }
}

void applySeparableBlur(
    const float* d_input,
    float* d_output,
    float* d_temp,
    const float* d_kernel1d,
    int width,
    int height,
    int radius
) {
    dim3 blockDim(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridDim(
        (width + blockDim.x - 1) / blockDim.x,
        (height + blockDim.y - 1) / blockDim.y
    );

    int sharedMemSize = (BLOCK_SIZE + 2 * radius) * sizeof(float);

    // Horizontal pass
    gaussianBlurHorizontalKernel<<<gridDim, blockDim, sharedMemSize>>>(
        d_input, d_temp, d_kernel1d, width, height, radius
    );

    // Vertical pass
    gaussianBlurVerticalKernel<<<gridDim, blockDim, sharedMemSize>>>(
        d_temp, d_output, d_kernel1d, width, height, radius
    );
}

void apply2DBlur(
    const float* d_input,
    float* d_output,
    const float* d_kernel2d,
    int width,
    int height,
    int radius
) {
    dim3 blockDim(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridDim(
        (width + blockDim.x - 1) / blockDim.x,
        (height + blockDim.y - 1) / blockDim.y
    );

    gaussianBlur2DKernel<<<gridDim, blockDim>>>(
        d_input, d_output, d_kernel2d, width, height, radius
    );
}

double benchmarkMethod(
    void (*method)(const float*, float*, float*, const float*, int, int, int),
    const float* d_input,
    float* d_output,
    float* d_temp,
    const float* d_kernel,
    int width,
    int height,
    int radius,
    int iterations,
    double* min_time,
    double* max_time
) {
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    double total_time = 0.0;
    *min_time = 1e9;
    *max_time = 0.0;

    // Warm-up
    method(d_input, d_output, d_temp, d_kernel, width, height, radius);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Benchmark
    for (int i = 0; i < iterations; i++) {
        CUDA_CHECK(cudaEventRecord(start));
        method(d_input, d_output, d_temp, d_kernel, width, height, radius);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float milliseconds = 0;
        CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start, stop));

        total_time += milliseconds;
        if (milliseconds < *min_time) *min_time = milliseconds;
        if (milliseconds > *max_time) *max_time = milliseconds;
    }

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    return total_time / iterations;
}

void verifyResults(const float* result1, const float* result2, int size) {
    double max_diff = 0.0;
    double sum_diff = 0.0;

    for (int i = 0; i < size; i++) {
        double diff = fabs(result1[i] - result2[i]);
        if (diff > max_diff) max_diff = diff;
        sum_diff += diff;
    }

    // Appropriate tolerance for float32:
    // - Machine epsilon for float32: ~1.19e-7
    // - With ~100 operations: expect ~1e-5 accumulated error
    // - Different computation order adds more error
    // - Conservative tolerance: 1e-6 (about 10x expected error)
    const double tolerance = 1e-6;

    printf("\n");
    printf("======================================================================\n");
    printf("Verification\n");
    printf("======================================================================\n");
    printf("\n\nResult comparison:\n");
    printf("  Max difference: %.2e\n", max_diff);
    printf("  Mean difference: %.2e\n", sum_diff / size);
    printf("  Results identical (tolerance=%.2e): %s\n",
           tolerance, max_diff < tolerance ? "True" : "False");
    printf("\n  Note: Tolerance is ~10000x float32 machine epsilon (1.19e-7)\n");
    printf("        This accounts for accumulated rounding errors and\n");
    printf("        different computation order between the two methods.\n");
}

void printSampleOutput(const float* image, int width, int height,
                      const char* title) {
    int cy = height / 2;
    int cx = width / 2;

    printf("\n%s (center 5x5):\n", title);
    for (int y = cy - 2; y <= cy + 2; y++) {
        printf("  ");
        for (int x = cx - 2; x <= cx + 2; x++) {
            printf("%.6f ", image[y * width + x]);
        }
        printf("\n");
    }
}

void printKernelInfo(const float* kernel1d, const float* kernel2d,
                     int radius, float sigma) {
    int size = 2 * radius + 1;

    printf("\n");
    printf("======================================================================\n");
    printf("Kernel Information (σ=%.1f, size=%dx%d)\n", sigma, size, size);
    printf("======================================================================\n");

    printf("\n\n1D Kernel (%d elements):\n  Values: ", size);
    float sum1d = 0.0f;
    for (int i = 0; i < size; i++) {
        printf("%.6f ", kernel1d[i]);
        sum1d += kernel1d[i];
    }
    printf("\n  Sum: %.10f (should be 1.0)\n", sum1d);

    printf("\n2D Kernel (%dx%d elements):\n", size, size);
    printf("  Center row: ");
    float sum2d = 0.0f;
    for (int i = 0; i < size; i++) {
        printf("%.6f ", kernel2d[radius * size + i]);
    }
    for (int i = 0; i < size * size; i++) {
        sum2d += kernel2d[i];
    }
    printf("\n  Sum: %.10f (should be 1.0)\n", sum2d);

    // Verify separability
    double max_sep_diff = 0.0;
    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            float outer_prod = kernel1d[i] * kernel1d[j];
            double diff = fabs(kernel2d[i * size + j] - outer_prod);
            if (diff > max_sep_diff) max_sep_diff = diff;
        }
    }

    printf("\n  Separability check:\n");
    printf("    Max difference between 2D and outer(1D,1D): %.2e\n", max_sep_diff);
    printf("    Kernel is separable: %s\n", max_sep_diff < 1e-10 ? "True" : "False");
}

// ============================================================================
// Main Function
// ============================================================================

int main(int argc, char** argv) {
    // Configuration
    int width = 1000;
    int height = 1000;
    float sigma = 2.0f;
    int radius = 5;  // kernel size = 11
    const char* pattern = "random";
    int iterations = 10;

    // Parse command line arguments
    for (int i = 1; i < argc; i++) {
        if (strcmp(argv[i], "--width") == 0 && i + 1 < argc) {
            width = atoi(argv[++i]);
        } else if (strcmp(argv[i], "--height") == 0 && i + 1 < argc) {
            height = atoi(argv[++i]);
        } else if (strcmp(argv[i], "--sigma") == 0 && i + 1 < argc) {
            sigma = atof(argv[++i]);
        } else if (strcmp(argv[i], "--kernel-size") == 0 && i + 1 < argc) {
            int ksize = atoi(argv[++i]);
            radius = ksize / 2;
        } else if (strcmp(argv[i], "--pattern") == 0 && i + 1 < argc) {
            pattern = argv[++i];
        } else if (strcmp(argv[i], "--iterations") == 0 && i + 1 < argc) {
            iterations = atoi(argv[++i]);
        }
    }

    int kernelSize = 2 * radius + 1;
    int imageSize = width * height;

    // Header
    printf("======================================================================\n");
    printf("Gaussian Blur: Separable vs Full 2D Convolution\n");
    printf("======================================================================\n");

    printf("\n\nConfiguration:\n");
    printf("  Image size: %d x %d\n", width, height);
    printf("  Kernel size: %d x %d\n", kernelSize, kernelSize);
    printf("  Sigma: %.1f\n", sigma);
    printf("  Pattern: %s\n", pattern);
    printf("  Benchmark iterations: %d\n", iterations);

    // Allocate host memory
    float* h_input = (float*)malloc(imageSize * sizeof(float));
    float* h_output_2d = (float*)malloc(imageSize * sizeof(float));
    float* h_output_sep = (float*)malloc(imageSize * sizeof(float));
    float* h_kernel1d = (float*)malloc(kernelSize * sizeof(float));
    float* h_kernel2d = (float*)malloc(kernelSize * kernelSize * sizeof(float));

    // Create test image
    printf("\nCreating test image (%s pattern)...\n", pattern);
    createTestImage(h_input, width, height, pattern);

    // Generate kernels
    generateGaussianKernel1D(h_kernel1d, radius, sigma);
    generateGaussianKernel2D(h_kernel2d, radius, sigma);

    // Print kernel info
    printKernelInfo(h_kernel1d, h_kernel2d, radius, sigma);

    // Allocate device memory
    float *d_input, *d_output_2d, *d_output_sep, *d_temp;
    float *d_kernel1d, *d_kernel2d;

    CUDA_CHECK(cudaMalloc(&d_input, imageSize * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_output_2d, imageSize * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_output_sep, imageSize * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_temp, imageSize * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_kernel1d, kernelSize * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_kernel2d, kernelSize * kernelSize * sizeof(float)));

    // Copy data to device
    CUDA_CHECK(cudaMemcpy(d_input, h_input, imageSize * sizeof(float),
                         cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_kernel1d, h_kernel1d, kernelSize * sizeof(float),
                         cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_kernel2d, h_kernel2d,
                         kernelSize * kernelSize * sizeof(float),
                         cudaMemcpyHostToDevice));

    // Benchmark Full 2D
    printf("\n");
    printf("======================================================================\n");
    printf("Benchmarking Full 2D Convolution\n");
    printf("======================================================================\n");

    double min_2d, max_2d;
    double mean_2d = benchmarkMethod(
        [](const float* in, float* out, float* tmp, const float* k,
           int w, int h, int r) {
            apply2DBlur(in, out, k, w, h, r);
        },
        d_input, d_output_2d, d_temp, d_kernel2d,
        width, height, radius, iterations, &min_2d, &max_2d
    );

    double throughput_2d = (imageSize / 1e6) / (mean_2d / 1000.0);
    int ops_2d = kernelSize * kernelSize;

    printf("\nFull 2D Convolution:\n");
    printf("  Time: %.3f ms\n", mean_2d);
    printf("  Range: [%.3f, %.3f] ms\n", min_2d, max_2d);
    printf("  Throughput: %.2f Mpixels/sec\n", throughput_2d);
    printf("  Operations/pixel: %d\n", ops_2d);

    // Copy 2D result to host
    CUDA_CHECK(cudaMemcpy(h_output_2d, d_output_2d, imageSize * sizeof(float),
                         cudaMemcpyDeviceToHost));

    // Benchmark Separable
    printf("\n");
    printf("======================================================================\n");
    printf("Benchmarking Separable Convolution\n");
    printf("======================================================================\n");

    double min_sep, max_sep;
    double mean_sep = benchmarkMethod(
        [](const float* in, float* out, float* tmp, const float* k,
           int w, int h, int r) {
            applySeparableBlur(in, out, tmp, k, w, h, r);
        },
        d_input, d_output_sep, d_temp, d_kernel1d,
        width, height, radius, iterations, &min_sep, &max_sep
    );

    double throughput_sep = (imageSize / 1e6) / (mean_sep / 1000.0);
    int ops_sep = 2 * kernelSize;

    printf("\nSeparable Convolution:\n");
    printf("  Time: %.3f ms\n", mean_sep);
    printf("  Range: [%.3f, %.3f] ms\n", min_sep, max_sep);
    printf("  Throughput: %.2f Mpixels/sec\n", throughput_sep);
    printf("  Operations/pixel: %d\n", ops_sep);

    // Copy separable result to host
    CUDA_CHECK(cudaMemcpy(h_output_sep, d_output_sep, imageSize * sizeof(float),
                         cudaMemcpyDeviceToHost));

    // Verify results
    verifyResults(h_output_2d, h_output_sep, imageSize);

    // Performance summary
    printf("\n");
    printf("======================================================================\n");
    printf("Performance Summary\n");
    printf("======================================================================\n");

    double speedup = mean_2d / mean_sep;
    double theoretical = (double)radius / 2.0;

    printf("\n\nSpeedup (Separable vs Full 2D):\n");
    printf("  Actual: %.2fx\n", speedup);
    printf("  Theoretical: %.2fx\n", theoretical);
    printf("  Efficiency: %.1f%%\n", (speedup / theoretical) * 100.0);

    // Sample outputs
    printf("\n");
    printf("======================================================================\n");
    printf("Sample Output Values (center 5x5 region)\n");
    printf("======================================================================\n");

    printSampleOutput(h_input, width, height, "\nInput");
    printSampleOutput(h_output_sep, width, height,
                     "\nOutput (both methods produce identical results)");

    // Cleanup
    free(h_input);
    free(h_output_2d);
    free(h_output_sep);
    free(h_kernel1d);
    free(h_kernel2d);

    CUDA_CHECK(cudaFree(d_input));
    CUDA_CHECK(cudaFree(d_output_2d));
    CUDA_CHECK(cudaFree(d_output_sep));
    CUDA_CHECK(cudaFree(d_temp));
    CUDA_CHECK(cudaFree(d_kernel1d));
    CUDA_CHECK(cudaFree(d_kernel2d));

    printf("\n");
    printf("======================================================================\n");
    printf("Completed Successfully!\n");
    printf("======================================================================\n");
    printf("\n");

    return 0;
}


Overwriting gaussian_blur_comparison_cuda.cu


In [28]:
!chmod +x ./run_comparison.sh
!./run_comparison.sh --width 1920 --height 1080 --kernel-size 15

Gaussian Blur Comparison Runner

✓ CUDA compiler found: Cuda compilation tools, release 12.8, V12.8.93
✓ Python found: Python 3.12.12
✓ NumPy and SciPy installed

Configuration:
  Image: 1920x1080
  Kernel: 15x15
  Sigma: 2.0
  Pattern: random
  Iterations: 10

Building and Running CUDA Version

Building CUDA application...
nvcc -arch=sm_75 -O3 --use_fast_math -Xcompiler -Wall -std=c++11 gaussian_blur_comparison_cuda.cu -o gaussian_blur_comparison_cuda
Built CUDA comparison application: gaussian_blur_comparison_cuda

Running CUDA version...
Gaussian Blur: Separable vs Full 2D Convolution


Configuration:
  Image size: 1920 x 1080
  Kernel size: 15 x 15
  Sigma: 2.0
  Pattern: random
  Benchmark iterations: 10

Creating test image (random pattern)...

Kernel Information (σ=2.0, size=15x15)


1D Kernel (15 elements):
  Values: 0.000436 0.002216 0.008765 0.027000 0.064769 0.121004 0.176059 0.199501 0.176059 0.121004 0.064769 0.027000 0.008765 0.002216 0.000436 
  Sum: 1.0000000000 (should